# Muestra instancias de sagemaker y su consumo de espacio de almacenamientos

In [1]:
# Muestra los archivos más pesados de la instancia actual
# !du -ah /home/ec2-user/SageMaker | sort -rh | head -n 20

In [2]:
# Elimina los archivos
# !rm -rf /home/ec2-user/SageMaker/Temp_Segmentacion/*

In [3]:
# Elimina los archivos dentro de la carpeta de papelera (.Trash)
# !rm -rf /home/ec2-user/SageMaker/.Trash-1000/*

# 1 Descargar Ventas y Visitas (aje-prd-analytics-artifacts-s3)

In [4]:

# import psutil
# import time
# import threading
# import matplotlib.pyplot as plt

# # Variables globales
# mem_usage_gb = []
# mem_usage_percent = []
# cpu_usage_gb = []
# cpu_usage_percent = []
# timestamps = []
# running = True  # Variable de control

# # Función de monitoreo en segundo plano
# def monitor_resources():
#     global running
#     start_time = time.time()
    
#     while running:
#         # Obtener información del sistema
#         mem_info = psutil.virtual_memory()
#         cpu_percent = psutil.cpu_percent(interval=0.5)  # Medir CPU sin bloquear
#         cpu_used_gb = (cpu_percent / 100) * (psutil.virtual_memory().total / (1024**3))  # Estimación del uso de CPU en GB
        
#         # Guardar datos
#         timestamp = time.time() - start_time
#         mem_usage_gb.append(mem_info.used / (1024 ** 3))  # RAM usada en GB
#         mem_usage_percent.append(mem_info.percent)  # RAM en %
#         cpu_usage_gb.append(cpu_used_gb)  # CPU en GB
#         cpu_usage_percent.append(cpu_percent)  # CPU en %
#         timestamps.append(timestamp)
        
#         # Imprimir valores en tiempo real
#         # print(f"Tiempo: {timestamp:.2f}s | RAM: {mem_info.percent:.2f}% ({mem_usage_gb[-1]:.2f} GB) | CPU: {cpu_percent:.2f}% ({cpu_used_gb:.2f} GB)")
        
#         time.sleep(1)  # Medir cada 1 segundo

# # Iniciar monitoreo en segundo plano
# monitor_thread = threading.Thread(target=monitor_resources, daemon=True)
# monitor_thread.start()

# print("Monitoreo de CPU y RAM iniciado en segundo plano...")

In [5]:
#!pip install 'awswrangler[redshift]'
#!pip install psycopg2-binary


!pip install --upgrade pip setuptools wheel
!pip install --upgrade pyarrow==17.0.0
!pip install "awswrangler[redshift]" --no-build-isolation
!pip install psycopg2-binary
!pip install openpyxl

In [6]:
import pandas as pd
import redshift_connector
import awswrangler as wr
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import os
import boto3
import io
import pytz
import gc

In [7]:
def comprobar_inputs():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/peru/"
    
    hoy = datetime.now(pytz.timezone("America/Lima")).date()
    errores = []

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" not in objetos:
        print("ERROR: No se encontraron archivos en la ruta especificada.")
        return

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        key = objeto["Key"]
        if key == "pedido_sugerido/data-v1/peru/pedidos_3meses_peru000":
            print(f"OJO: El key {key} no pasa validacion.")
            continue
        print(key)
        # Omitir "carpetas" en S3
        if key.endswith("/"):
            continue

        last_modified = objeto["LastModified"].date()
        size_kb = objeto["Size"] / 1024  # Convertir a KB

        # Verificar si el objeto tiene contenido real (FULL_OBJECT)
        if objeto["Size"] == 0:
            errores.append(f"ERROR: El archivo {key} está vacío.")

        # Verificar si el archivo ha sido modificado hoy
        if last_modified != hoy:
            errores.append(f"ERROR: El archivo {key} no ha sido modificado hoy ({hoy}), su ultima fecha de modificacion fue {last_modified}.")

        # Verificar si el tamaño del archivo es menor a 1 KB
        if size_kb < 1:
            errores.append(f"ALERTA: El archivo {key} tiene un tamaño menor a 1 KB ({size_kb:.2f} KB).")

    # Mostrar los errores y lanzar una excepción si es necesario
    if errores:
        for error in errores:
            print(error)
        raise ValueError("Se encontraron problemas con los archivos en S3.")
    else:
        print("Todo bien :D")

In [8]:
comprobar_inputs()

pedido_sugerido/data-v1/peru/
pedido_sugerido/data-v1/peru/D_stock_pe.csv
pedido_sugerido/data-v1/peru/maestro_productos_peru000
OJO: El key pedido_sugerido/data-v1/peru/pedidos_3meses_peru000 no pasa validacion.
pedido_sugerido/data-v1/peru/ventas_peru000
pedido_sugerido/data-v1/peru/ventas_peru001
pedido_sugerido/data-v1/peru/visitas_peru000
Todo bien :D


## 1.0 Descargar Maestro de productos

In [9]:
%%time
# PARA 2024
query = f"""select cod_articulo_magic, desc_articulo from 
    comercial_peru.dim_producto
    where estado='A' and instancia='PE';
    """

con = wr.data_api.redshift.connect(
    cluster_id="dwh-cloud-storage-salesforce-prod",
    database="dwh_prod",
    db_user="dwhuser",
)
maestro_prod = wr.data_api.rds.read_sql_query(query, con)

CPU times: user 1.61 s, sys: 91.5 ms, total: 1.7 s
Wall time: 5.07 s


In [10]:
maestro_prod = maestro_prod[["cod_articulo_magic", "desc_articulo"]].groupby(["cod_articulo_magic"]).head(1).reset_index(
    drop=True
)

In [11]:
maestro_prod

,cod_articulo_magic,desc_articulo
0,10887,BOTAS DE SEGURIDAD CON PUNTA D
1,13313,BOTAS DE PVC BLANCO TALLA-38 C
2,13353,EXTINTOR DE PQS 12 KG
3,13790,AZUCAR EN BOLSA X KILO
4,14033,CAMIONCITOS X 4
...,...,...
11426,438,DEST POLVO FLORAL BOLSA DETERGENTE 4000 GRS 4
11427,463,DEST LAVAVAJILLAS LIMON BOTELLA PET + ESPONJA ...
11428,467,D GUSSTO COCOA BOLSA 13 GRS 192
11429,12001350,CARRO DE HERRAMIENTAS


In [12]:
maestro_prod.to_csv("Input/PE_maestro_productos.csv", index=False)

In [13]:
rutas_peru = [
10000,10001,10002,10003,10004,10005,10006,10007,10008,10009,10010,10011,10012,10013,10014,10015,10016,10017,10018,10019,
10020,10021,10022,10023,10024,10025,10026,10027,10028,10029,10030,10031,10032,10033,10034,10035,10036,10037,10038,10039,
10040,10041,10042,10043,10044,10045,10046,10047,10048,10049,10050,10051,10052,10053,10054,10055,10056,10057,10059,10061,
10062,10066,10067,10060,10063,10064,10065,10058,19001,13479,13486,13481,13485,13478,13487,13482,13483,13484,10400,10401,
10402,10403,10404,10405,10406,10407,10408,10409,10410,10576,10413,10578,10577,10579,10580,10414,10415,10416,10420,10421,
10422,10417,10418,10419,10426,10427,10428,10429,10430,10431,10432,10433,10434,10435,10423,10425,10444,10424,10445,10452,
10446,10447,10448,10449,10450,10451,10461,10465,10459,10460,10462,10464,10463,10466,10467,10473,10468,10469,10470,10472,
10471,10591,10600,10601,10602,10603,10604,10605,10606,10607,10608,10609,10610,10611,10612,10613,10614,10615,10616,10617,
10618,10619,10620,10621,10622,10623,10624,10625,10626,10627,10628,10629,10630,10631,10632,10633,10634,10635,10636,10637,
10638,10639,10640,10641,10642,10643,10644,10645,10646,10647,10648,10649,10650,10660,10651,10652,10653,10654,10655,10656,
10657,10658,10659,10661,10662,10663,10664,10665,10666,10667,10668,10669,10670,10730,10731,10800,10805,10806,10801,10807,
10802,10808,10803,10848,10804,10839,10812,10843,10838,10813,10846,10814,10815,10816,10817,10820,10822,10823,10825,10826,
10827,10828,10830,10831,10832,10836,10837,10840,10841,10842,11000,11001,11002,11003,11004,11005,11006,11007,11008,11009,
11010,11011,11012,11013,11014,11015,11017,11018,11019,11020,11021,11023,11024,11025,11026,11200,11201,11202,11203,11204,
11205,11206,11207,11215,11216,11217,11218,11214,11222,11224,11225,11226,11210,11211,11212,11213,11219,11220,11221,11223,
11400,11401,11402,11403,11404,11405,11410,11406,11407,11408,11409,11411,11412,11413,11414,11600,11601,11602,11603,11604,
11605,11606,11607,11608,11609,11610,11611,11612,11613,11614,11615,11616,11617,11627,11618,11620,11621,11622,11623,11624,
11625,11626,11619,11800,11801,11802,11803,11804,11805,11806,11807,11808,11809,11810,11811,11812,11813,11814,11815,11816,
11817,11818,11819,12000,12001,12002,12003,12004,12005,12006,12007,12008,12009,12010,12011,12012,12013,12200,12201,12202,
12203,12204,12205,12206,12207,12208,12209,12210,12211,12212,12213,12214,12215,12216,12217,12218,12219,12220,12221,12222,
12223,12230,12234,12224,12229,12235,12225,12226,12227,12228,12231,12232,12233,12236,12237,12238,12240,12239,12405,12400,
12401,12402,12403,12404,12500,12501,12502,12503,12504,12505,13530,13531,13532,13533,13535,13534,13536,13400,13401,13402,
13403,13404,13405,13406,13414,13415,13409,13416,13417,13410,13411,13412,13413,13418,13421,13420,13419,13422,13423,13424,
13425,13460,13461,13462,13463,13464,13465,13466,13467,13468,13469,13470,13471,13472,13473,13474,13475,13476,12973,12954,
12956,12969,12961,12958,12950,12953,12963,12959,12962,12951,12952,12965,12957,12972,12955,12974,12975,12960,12976,12964,
12967,12970,13543,13540,13544,13541,13545,13542,13546,13547,13539,13548,13549,13550,13551,13552,13553,13554,13555,13556,
13557,13558,13565,13559,13566,13560,13561,13567,13568,13562,13564,13563,13569,13570,13574,13571,13572,13573,13575,13576,
13577,13584,13581,13579,13585,13582,13586,13583,13578,13580,13587,13588,13589,13594,13591,13593,13592,13595,13596,13597,
13598,13654,13624,13620,13621,13623,13625,13622,13655,13656,13651,13653,13650,13652,13800,13801,13802,13803,13804,13805,
13806,13807,13808,13809,13813,13811,13814,13812,13815,13816,13817,13818,13819,13820,13850,13851,13852,13853,13854,13855,
14101,14100,14102,14103,14104,14105,14106,14107,14108,14109,14110,14111,14112,14113,14114,14115,14116,14117,14118,14119,
14120,14121,14450,14457,14461,14451,14459,14452,14460,14453,14454,14456,14455,14458,14465,14462,14466,14464,14467,14463,
14468,14509,14500,14502,14501,14503,14504,14505,14507,14506,14508,14511,14512,14510,14519,14513,14515,14514,14518,14516,
14524,14517,14525,14522,14523,14521,14609,14605,14606,14610,14603,14602,14604,14607,14608]

In [14]:
len(rutas_peru)

675

In [15]:
rutas_peru = list(set(rutas_peru)) # quitamos rutas duplicadas
len(rutas_peru)

675

In [16]:
clientes_ruta_test_peru = []

In [17]:
def descargar_visitas():
    # Guardar clientes de las rutas
    global clientes_ruta_test_peru

    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/peru/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0 and objeto["Key"].split("/")[-1] == "visitas_peru000":
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "visitas_peru000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[(df["cod_ruta"].isin(rutas_peru))]
                # Guardamos estos clientes para descargarlos en ventas (por si cambiaron de ruta)
                clientes_ruta_test_peru = df["codigo_cliente__c"].unique()
                nombre_csv = f"Input/{nombre_archivo}.parquet"
                df["personalizacion"] = df["personalizacion"].astype(str)
                df.to_parquet(nombre_csv, index=False)

In [18]:
%%time
descargar_visitas()

{'ResponseMetadata': {'RequestId': 'AWM5XX0AYVN1M375', 'HostId': 'Zfl3qOYyaL5rzudgr+EYnGgersD3B1ZiKeRviW83NycIuS9uuemQ2h3KFwzeSDwmKkgaILvFZdI=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'Zfl3qOYyaL5rzudgr+EYnGgersD3B1ZiKeRviW83NycIuS9uuemQ2h3KFwzeSDwmKkgaILvFZdI=', 'x-amz-request-id': 'AWM5XX0AYVN1M375', 'date': 'Tue, 31 Mar 2026 20:17:17 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/peru/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 31, 23, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/peru/D_stock_pe.csv', 'LastModified': datetime.datetime(2026, 3, 31, 14, 8, 22, tzinfo=tzlocal()), 'ETag': '"c3f279814df7d2d8396206358f7f12a1"', 'ChecksumAlgorithm': ['CRC32'], 'ChecksumType': 'FULL_OBJECT', 'Size': 546935,

In [19]:
#1m47s

In [20]:
def descargar_ventas():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/peru/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
    print(objetos)

    # Creamos DF que juntará los 2 archivos
    final_csv = pd.DataFrame()
    df_cols = ['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0:
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]
            
            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "ventas_peru000":
                # Descargar el archivo en memoria
                response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
                content = response["Body"].read()
                
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[df_cols]
                df = df[
                    (df["cod_ruta"].isin(rutas_peru))
                    | (df["cod_cliente"].isin(clientes_ruta_test_peru))
                ]
                final_csv = pd.concat([final_csv, df], ignore_index=True)

            elif nombre_archivo == "ventas_peru001":
                # Descargar el archivo en memoria
                response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
                content = response["Body"].read()

                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[df_cols]
                df = df[
                    (df["cod_ruta"].isin(rutas_peru))
                    | (df["cod_cliente"].isin(clientes_ruta_test_peru))
                ]
                final_csv = pd.concat([final_csv, df], ignore_index=True)
            elif nombre_archivo == "ventas_peru002":
                # Descargar el archivo en memoria
                response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
                content = response["Body"].read()

                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[df_cols]
                df = df[
                    (df["cod_ruta"].isin(rutas_peru))
                    | (df["cod_cliente"].isin(clientes_ruta_test_peru))
                ]
                final_csv = pd.concat([final_csv, df], ignore_index=True)
            else:
                continue
    nombre_csv = f"Input/ventas_peru000_all.parquet"
    final_csv.to_parquet(nombre_csv, index=False)

In [21]:
%%time
descargar_ventas()

{'ResponseMetadata': {'RequestId': '634YRNYAHDHCFHZW', 'HostId': 'OXM2LOR/om39EPO0ne3gKZBE2UUVVTh1z+0LyE36pKVNHaf2Q4kN8iFQSb0yGHY/PLWeDFFmmd3+YCIPrl1Gh1Sbczw7fZQa', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'OXM2LOR/om39EPO0ne3gKZBE2UUVVTh1z+0LyE36pKVNHaf2Q4kN8iFQSb0yGHY/PLWeDFFmmd3+YCIPrl1Gh1Sbczw7fZQa', 'x-amz-request-id': '634YRNYAHDHCFHZW', 'date': 'Tue, 31 Mar 2026 20:17:19 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/peru/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 31, 23, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/peru/D_stock_pe.csv', 'LastModified': datetime.datetime(2026, 3, 31, 14, 8, 22, tzinfo=tzlocal()), 'ETag': '"c3f279814df7d2d8396206358f7f12a1"', 'ChecksumAlgorithm': ['CRC32'], 'Check

## 1.1 Juntar VENTAS y VISITAS

In [22]:
import pandas as pd
from datetime import datetime, timedelta

In [23]:
pan_ventas = pd.read_parquet("Input/ventas_peru000_all.parquet")
pan_visitas = pd.read_parquet("Input/visitas_peru000.parquet")

In [24]:
pan_visitas = pan_visitas[pan_visitas.codigo_canal__c==2].reset_index(drop=True)

In [25]:
pan_ventas["cod_articulo_magic"] = pan_ventas["id_producto"].str.split("|").str[-1]
pan_ventas["cod_articulo_magic"] = pan_ventas["cod_articulo_magic"].astype(int)

In [26]:
pan_ventas.shape

(25680703, 26)

In [27]:
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print(hoy)
# Filtrar solo las filas con la fecha de hoy
pan_ventas = pan_ventas[pan_ventas['fecha_proceso'] == hoy]

20260331


In [28]:
pan_ventas.shape

(25680703, 26)

In [29]:
pan_ventas.desc_marca = pan_ventas.desc_marca.str.strip()

In [30]:
# Eliminamos TRES CRUCES de todas las sucursales que no sean de LIMA
pan_ventas=pan_ventas[
    (pan_ventas['desc_sucursal'].isin(['LIMA-CALLAO', 'PUENTE PIEDRA', 'VILLA EL SALVADOR', 'HUACHIPA'])) |  # Mantener todo de Lima Norte y Sur, centro y callao
    (~pan_ventas['desc_marca'].isin(['TRES CRUCES']))  # En otras sucursales, eliminar "TRES CRUCES"
].reset_index(drop=True)

In [31]:
# Quitar SKUS de Lima Centro
rutas_excluir = [10673,10600,10601,10602,10603,10604,10605,10606,10607,
10608,10609,10610,10611,10612,10613,10614,10615,
10616,10617,10618,10619,10620,10621,10622,10623,
10624,10625,10626,10627,10628,10629,10630,10631,
10632,10633,10634,10635,10636,10637,10638,10639,
10640,10641,10642,10643,10644,10645,10646,10647,
10648,10649,10650,10651,10652,10653,10654,10655,
10656,10657,10658,10659,10660,10661,10662,10663,
10664,10665,10666,10667,10668,10669,10670]
skus_excluir = [622396,621807,622807,622394,622395,
                608962,623130,623679,622356,699892,
                623456,621858,621851,699549,622836,
                621994,621996,623262,622957,622163,
                622956,621807,622807,623679,699892,
                622836,621994,621996,622956,622396,
                621807,622807,622394,608962,623130,
                623679,623456,621858,621851,699549,
                622836,621994,621996]
# Filtramos manteniendo solo las filas que NO cumplan ambas condiciones
pan_ventas = pan_ventas[~(pan_ventas['cod_ruta'].isin(rutas_excluir) & pan_ventas['cod_articulo_magic'].isin(skus_excluir))]

In [32]:
pan_ventas.shape

(25669460, 26)

In [33]:
# Obtener la fecha y año actual
current_date = datetime.now(pytz.timezone("America/Lima"))
# Formatear la fecha en el formato "YYYYMM"
formatted_date = current_date.strftime("%Y-%m-%d")
formatted_date

'2026-03-31'

In [34]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Peru/Input/visitas_peru000_{formatted_date}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(pan_visitas, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Peru/Input/visitas_peru000_2026-03-31.csv'],
 'partitions_values': {}}

In [35]:
# Leemos archivo de marcas y skus especificos para peru
skus_a_usar = pd.read_excel("Input/pe_ps_marcas_skus_a_usar.xlsx", sheet_name="Hoja1")

In [36]:
skus_a_usar.cod_articulo_magic.nunique()

621

In [37]:
pan_ventas.cod_articulo_magic.nunique()

543

In [38]:
pan_ventas.desc_marca.nunique()

43

In [39]:
# Nos quedamos solo con algunas SKUs especificas
pan_ventas = pan_ventas[
    (pan_ventas.cod_articulo_magic.isin(skus_a_usar.cod_articulo_magic.unique()))
    | (pan_ventas.desc_marca == "TRES CRUCES")
].reset_index(drop=True)

In [40]:
pan_ventas.desc_marca.nunique()

20

In [41]:
pan_ventas.cod_articulo_magic.nunique()

316

In [42]:
pan_ventas["cod_ruta"].nunique()

904

### 1.1.1 Filtrar rutas de Lima para Peru

In [43]:
rutas = pan_ventas["cod_ruta"].unique()

In [44]:
pan_ventas.shape

(22095032, 26)

In [45]:
# PAN VENTAS
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["id_cliente"] = (
    "PE"
    + "|"
    + pan_ventas["cod_compania"].astype(str)
    + "|"
    + pan_ventas["cod_cliente"].astype(str)
)

# PAN VISITAS
pan_visitas["compania__c"] = (
    pan_visitas["compania__c"].astype(str).apply(lambda x: str((x)).rjust(4, "0"))
)
pan_visitas["id_cliente"] = (
    "PE"
    + "|"
    + pan_visitas["compania__c"].astype(str)
    + "|"
    + pan_visitas["codigo_cliente__c"].astype(str)
)

In [46]:
# Nos quedamos con las visitas de clientes tradicional
pan_visitas = pan_visitas[pan_visitas.codigo_canal__c == 2].reset_index(drop=True)

In [47]:
pan_ventas.shape

(22095032, 26)

In [48]:
# Eliminar Duplicados y quedarse con la ultima visita más reciente
visita_default_semana_atras = (datetime.now(pytz.timezone("America/Lima")) - timedelta(days=7)).strftime("%Y-%m-%d")
pan_visitas["ultima_visita"] = pan_visitas["ultima_visita"].fillna(
    visita_default_semana_atras
)
pan_visitas = (
    pan_visitas.sort_values(["id_cliente", "ultima_visita"], ascending=False)
    .groupby("id_cliente")
    .head(1)
)

In [49]:
# Combina ventas y visitas (nos quedamos con ruta y módulo de visitas si es nulo, usamos el valor de ventas)
df_merged = pd.merge(
    pan_ventas,
    pan_visitas[
        [
            "id_cliente",
            "dias_de_visita__c",
            "periodo_de_visita__c",
            "ultima_visita",
            "cod_ruta",
            "cod_modulo",
            "eje_potencial__c",
        ]
    ],
    on="id_cliente",
    # how='outer',  # Este es un merge 'outer', para conservar todos los clientes de ambos DataFrames.
    how="inner",
    suffixes=(
        "_df1",
        "_df2",
    ),  # Añadimos sufijos para diferenciar las columnas que se solapan
)

# Reemplazar los valores nulos de 'cod_ruta_df2' con los valores de 'cod_ruta_df1', y viceversa
# Si 'cod_ruta_df2' es nulo, se usará el valor de 'cod_ruta_df1'. Si ambos son nulos, se mantendrá nulo.
df_merged["cod_ruta"] = df_merged["cod_ruta_df2"].combine_first(
    df_merged["cod_ruta_df1"]
)
df_merged["cod_modulo"] = df_merged["cod_modulo_df2"].combine_first(
    df_merged["cod_modulo_df1"]
)

# Eliminar las columnas innecesarias ('cod_ruta_df1', 'cod_ruta_df2', 'cod_modulo_df1', 'cod_modulo_df2')
df_merged = df_merged.drop(
    columns=["cod_ruta_df1", "cod_ruta_df2", "cod_modulo_df1", "cod_modulo_df2"]
)

In [50]:
pan_ventas = df_merged.copy()
# Liberar memoria del df_merged
df_merged = None 
gc.collect()

2222

In [51]:
pan_ventas["cod_cliente"] = pan_ventas["cod_cliente"].astype(str)
pan_ventas["cod_ruta"] = pan_ventas["cod_ruta"].astype(int)
pan_ventas["cod_modulo"] = pan_ventas["cod_modulo"].astype(int)

In [52]:
pan_ventas["id_cliente"].nunique()

181835

In [53]:
pan_ventas.cod_ruta.nunique()

666

In [54]:
pan_ventas.groupby("cod_sucursal").agg(
    {"cod_articulo_magic": "nunique", "id_cliente": "nunique", "cod_ruta": "nunique"}
)

,cod_articulo_magic,id_cliente,cod_ruta
cod_sucursal,,,
1,212,18062,66
2,206,7582,27
3,189,17459,57
4,181,7481,31
5,151,3313,16
6,214,28891,101
7,193,7603,32
8,183,6228,19
9,216,4455,26


# 2 Ajustar DF de clientes: Se filtrará a los clientes que tienen visita al día siguiente

## 2.2 Combinar Ventas con Subgiro, Descripcion de productos, Segmento del cliente y Fecha de creacion

In [55]:
# Leer Archivos
# CAMBIAR DEPORADE POR SPORADE
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].str.strip()
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].replace({"DEPORADE": "SPORADE"})

pan_prod = pd.read_csv("Input/PE_maestro_productos.csv")

# Crear ID_CLIENTE
# PAN_VENTAS
pan_ventas["cod_sucursal"] = pan_ventas["cod_sucursal"].astype(str)

# PAN SEGMENTOS
pan_ventas = pan_ventas.rename(columns={"eje_potencial__c": "new_segment"})
# mapear valores
mapping = {
    "S1": "BLINDAR",
    "S2": "DESARROLLAR",
    "S4": "MANTENER",
    "S5": "OPTIMIZAR",
}

pan_ventas["new_segment"] = pan_ventas["new_segment"].map(mapping)
pan_ventas["new_segment"] = pan_ventas["new_segment"].fillna("OPTIMIZAR")

In [56]:
pan_ventas = pd.merge(
    pan_ventas,
    pan_prod[["cod_articulo_magic", "desc_articulo"]],
    how="left",
    on="cod_articulo_magic",
)

pan_ventas["mes"] = pd.to_datetime(pan_ventas["fecha_liquidacion"]).dt.strftime(
    "%Y-%m-01"
)

In [57]:
pan_ventas.to_parquet("Processed/ventas_peru_12m.parquet", index=False)

## 2.3 Seleccionando SKUs que no se commpran en los ultimos 3 dias
La variable **sucursal_sku** contiene todos los SKUs disponibles por sucursal

In [58]:
pan_ventas[["cod_compania","cod_sucursal"]].drop_duplicates().to_csv("pe_compa_sucursal_2025-07-30.csv",index=False)

In [59]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [60]:
pan_ventas.id_cliente.nunique()

181835

In [61]:
sucursales = pan_ventas["cod_sucursal"].unique()
sucursales

array(['40', '79', '9', '26', '7', '1', '28', '2', '3', '4', '5', '6',
       '18', '19', '8', '10', '14', '20', '21', '22', '25'], dtype=object)

In [62]:
pan_ventas.cod_articulo_magic.nunique()

300

In [63]:
# Obtener la fecha actual
fecha_actual = datetime.now(pytz.timezone("America/Lima")).date()
# Lista para almacenar las fechas
last_3_days = []
dias_atras = 3
sucursal_sku = {}
# Obtener las fechas de los últimos 3 días
for i in range(1, dias_atras + 1):
    fecha = fecha_actual - timedelta(days=i)
    fecha_formato = fecha.strftime("%Y-%m-%d")
    last_3_days.append(fecha_formato)

In [64]:
last_3_days

['2026-03-30', '2026-03-29', '2026-03-28']

In [65]:
for sucursal in sucursales:
    temp = pan_ventas[pan_ventas["cod_sucursal"] == sucursal]
    # sku_fecha=temp[["cod_articulo_magic","fecha_liquidacion"]].sort_values(["cod_articulo_magic","fecha_liquidacion"]).drop_duplicates().reset_index(drop=True)
    # sku_fecha=sku_fecha.groupby("cod_articulo_magic")[["cod_articulo_magic","fecha_liquidacion"]].tail(3)
    # #Listando SKUs comprados al menos una vez en los ultimos 3 dias
    # sku_in_last_3_days=sku_fecha[sku_fecha.fecha_liquidacion.isin(last_3_days)]["cod_articulo_magic"].unique()
    # print(f"Sucursal {sucursal} eliminando SKUs:",sku_fecha[(~sku_fecha["cod_articulo_magic"].isin(sku_in_last_3_days))].reset_index(drop=True)["cod_articulo_magic"].sort_values().unique())
    # #SKU COMPRADO AL MENOS una vez en los ultimos 3 dias
    # sku_fecha=sku_fecha[sku_fecha["cod_articulo_magic"].isin(sku_in_last_3_days)].reset_index(drop=True)
    # temp=temp[temp["cod_articulo_magic"].isin(sku_fecha["cod_articulo_magic"].unique())].reset_index(drop=True)
    sucursal_sku[sucursal] = temp["cod_articulo_magic"].sort_values().unique()

In [66]:
pan_ventas.groupby("cod_sucursal")["id_cliente"].nunique()

cod_sucursal
1     18062
10     2863
14     7699
18    14674
19    19032
2      7582
20     4406
21     2064
22    14767
25     4291
26    11549
28     1548
3     17459
4      7481
40       41
5      3313
6     28891
7      7603
79       11
8      6228
9      4455
Name: id_cliente, dtype: int64

## 2.4 Filtrar clientes a visitar mañana

In [67]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [68]:
pan_ventas.id_cliente.nunique()

181835

In [69]:
# Conteo de clientes unicos por dia de visita
pan_ventas.groupby("dias_de_visita__c").id_cliente.nunique()

dias_de_visita__c
1               7305
1;2;3;4;5;6        5
1;3              335
1;3;5           1673
1;3;6              1
1;4            43317
1;5              614
2               8220
2;4               85
2;4;6           2884
2;5            43385
2;6              213
3               7401
3;5              388
3;6            42759
3;7              261
4               6992
4;5               70
4;6              179
5               7921
6               7231
7                596
Name: id_cliente, dtype: int64

In [70]:
data_test = (
    pan_ventas[
        ["id_cliente", "dias_de_visita__c", "periodo_de_visita__c", "ultima_visita"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
data_test["ultima_visita"] = pd.to_datetime(
    data_test["ultima_visita"], format="%Y-%m-%d"
)
# Calcula la diferencia en días entre la fecha de cada fila y la fecha de hoy
fecha_actual = datetime.now() + timedelta(days=0)
data_test["dias_pasados"] = (fecha_actual - data_test["ultima_visita"]).dt.days
fecha_actual

datetime.datetime(2026, 3, 31, 20, 29, 56, 260846)

In [71]:
# Obtener el día de la semana actual (1 para lunes, 2 para martes, ..., 7 para domingo)
dia_actual = datetime.now(pytz.timezone("America/Lima")).weekday() + 1

# Si hoy es domingo (7), el día siguiente es lunes (1), de lo contrario, es el siguiente día al actual
if dia_actual == 6:
    dia_siguiente = 7
    # dia_siguiente=6
else:
    dia_siguiente = (dia_actual + 1) % 7
    # dia_siguiente = (dia_actual ) % 7

# Filtrar los clientes que serán visitados mañana
clientes_a_visitar_manana = data_test[
    data_test["dias_de_visita__c"]
    .astype(str)
    .apply(lambda x: str(dia_siguiente) in x.split(";"))
].reset_index(drop=True)
print("dia_actual", dia_actual)
print("dia_siguiente", dia_siguiente)

dia_actual 2
dia_siguiente 3


In [72]:
# Definir condiciones de filtro
condicion_f1 = clientes_a_visitar_manana["periodo_de_visita__c"] == "F1"
condicion_f2 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F2") & (
    clientes_a_visitar_manana["dias_pasados"] > 13
)
condicion_f3 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F3") & (
    clientes_a_visitar_manana["dias_pasados"] > 20
)
condicion_f4 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F4") & (
    clientes_a_visitar_manana["dias_pasados"] > 27
)

# Aplicar las condiciones de filtro
clientes_a_visitar_manana = clientes_a_visitar_manana[
    condicion_f1 | condicion_f2 | condicion_f3 | condicion_f4
].reset_index(drop=True)

In [73]:
clientes_a_visitar_manana.isnull().sum()

id_cliente              0
dias_de_visita__c       0
periodo_de_visita__c    0
ultima_visita           0
dias_pasados            0
dtype: int64

In [74]:
clientes_a_visitar_manana.periodo_de_visita__c.unique()

array(['F1', 'F2'], dtype=object)

In [75]:
clientes_a_visitar_manana.dias_de_visita__c.unique()

array(['3;6', '3', '3;5', '1;3;5', '1;3', '1;3;6', '3;7', '1;2;3;4;5;6'],
      dtype=object)

In [76]:
pan_ventas.groupby("cod_ruta")["dias_de_visita__c"].unique().reset_index()

,cod_ruta,dias_de_visita__c
0,10000,"[3;6, 1;4, 2;5, 5, 2]"
1,10001,"[2;5, 1;4, 3;6, 1, 4, 3, 6]"
2,10002,"[1;4, 2;5, 3;6, 4, 2, 3, 6]"
3,10003,"[1;4, 3;6, 2;5, 3, 2]"
4,10004,"[3;6, 2;5, 1;4, 3]"
...,...,...
661,14605,"[1;4, 3;6, 2;5]"
662,14606,"[3;6, 1;4, 2;5, 3, 2, 1]"
663,14607,"[2;5, 3;6, 1;4, 1]"
664,14608,"[1;4, 3;6, 2;5]"


In [77]:
pan_ventas = pan_ventas[
    pan_ventas["id_cliente"].isin(clientes_a_visitar_manana["id_cliente"])
].reset_index(drop=True)
pan_ventas

,id_cliente,id_sucursal,id_producto,fecha_liquidacion,cod_zona,cant_cajafisicavta,cant_cajaunitvta,imp_netovta,cod_compania,desc_compania,...,fecha_proceso,cod_articulo_magic,dias_de_visita__c,periodo_de_visita__c,ultima_visita,new_segment,cod_ruta,cod_modulo,desc_articulo,mes
0,PE|0012|150908,PE|0012|40,PE|0012|608469,2025-11-17,1444,1.0,0.200000,8.4712,0012,COMERCIALIZADORA BETANIA S.A.C,...,20260331,608469,3;6,F1,2025-12-27,OPTIMIZAR,12239,13418,CIELO AGUA SIN GAS PET NO RETORNABLE 1000 ml 6...,2025-11-01
1,PE|0012|150908,PE|0012|40,PE|0012|608469,2025-04-21,1444,1.0,0.200000,8.4747,0012,COMERCIALIZADORA BETANIA S.A.C,...,20260331,608469,3;6,F1,2025-12-27,OPTIMIZAR,12239,13418,CIELO AGUA SIN GAS PET NO RETORNABLE 1000 ml 6...,2025-04-01
2,PE|0015|67094,PE|0015|09,PE|0015|599411,2026-01-29,1360,1.0,0.200000,10.6838,0015,AJEPER DEL ORIENTE S.A.,...,20260331,599411,3;6,F1,2026-03-28,BLINDAR,11802,12708,ORO ORO PET NO RETORNABLE 400 ML 15,2026-01-01
3,PE|0015|59798,PE|0015|09,PE|0015|699364,2025-06-29,1361,1.0,0.116000,12.8500,0015,AJEPER DEL ORIENTE S.A.,...,20260331,699364,3;6,F1,2026-03-28,MANTENER,11818,12756,PULP FORTIHIERRO DURAZNO TETRA PAK 145 ML 24 MC,2025-06-01
4,PE|0015|57984,PE|0015|09,PE|0015|522166,2025-08-28,1361,1.0,0.333333,8.5000,0015,AJEPER DEL ORIENTE S.A.,...,20260331,522166,3;6,F1,2026-03-28,OPTIMIZAR,11815,12747,VIDA AGUA PET NO RETORNABLE 500 ML 20,2025-08-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6547417,PE|1003|218978,PE|1003|28,PE|1003|608469,2026-03-05,1490,1.0,0.200000,8.4746,1003,COMERCIALIZADORA CODISAL S.A.C.,...,20260331,608469,3;6,F1,2026-03-28,OPTIMIZAR,12502,13708,CIELO AGUA SIN GAS PET NO RETORNABLE 1000 ml 6...,2026-03-01
6547418,PE|1003|218978,PE|1003|28,PE|1003|608469,2026-02-09,1490,2.0,0.400000,16.9492,1003,COMERCIALIZADORA CODISAL S.A.C.,...,20260331,608469,3;6,F1,2026-03-28,OPTIMIZAR,12502,13708,CIELO AGUA SIN GAS PET NO RETORNABLE 1000 ml 6...,2026-02-01
6547419,PE|1003|217748,PE|1003|28,PE|1003|622960,2026-02-02,1490,3.0,0.160000,27.9661,1003,COMERCIALIZADORA CODISAL S.A.C.,...,20260331,622960,3;6,F1,2026-03-28,MANTENER,12500,13702,DEST DETERGENTE POLVO FLORAL BOLSA 800 ML 2,2026-02-01
6547420,PE|1003|217748,PE|1003|28,PE|1003|622960,2026-03-02,1490,3.0,0.160000,27.9661,1003,COMERCIALIZADORA CODISAL S.A.C.,...,20260331,622960,3;6,F1,2026-03-28,MANTENER,12500,13702,DEST DETERGENTE POLVO FLORAL BOLSA 800 ML 2,2026-03-01


In [78]:
pan_ventas.periodo_de_visita__c.unique()

array(['F1', 'F2'], dtype=object)

In [79]:
pan_ventas.dias_de_visita__c.unique()

array(['3;6', '3', '3;5', '1;3;5', '1;3', '1;3;6', '3;7', '1;2;3;4;5;6'],
      dtype=object)

In [80]:
pan_ventas.id_cliente.nunique()

52387

In [81]:
pan_ventas.cod_compania.unique()

array(['0012', '0015', '1003', '0016', '0018', '0020'], dtype=object)

In [82]:
pan_ventas.to_parquet("Processed/peru_ventas_manana.parquet", index=False)

# 3. Pre Procesamiento

In [83]:
import pandas as pd
import numpy as np
from datetime import datetime

In [84]:
df_ventas = pd.read_parquet("Processed/peru_ventas_manana.parquet")

In [85]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(
    df_ventas["fecha_liquidacion"], format="%Y-%m-%d"
)
df_ventas["desc_marca"] = df_ventas["desc_marca"].str.strip()
df_ventas["desc_categoria"] = df_ventas["desc_categoria"].str.strip()

## 3.1 Mapeo de Pesos por Giro
Los pesos representan el orden en el que se recomendara la marca para cada cliente dependiendo del Giro

In [86]:
# Mapeo de pesos para ordenar las recomendaciones finales segun su importancia por giro
mapeo_pesos = {}
mapeo_diccionario = {}

In [87]:
df_ventas.desc_categoria.unique()

array(['AGUA', 'GASEOSAS', 'NECTAR', 'BEBIDAS DEPORTIVAS', 'ENERGIZANTE',
       'JUGOS LIGEROS', 'CAFE', 'CONSERVA DE PESCADO', 'SUAVIZANTES',
       'ACEITE', 'PASTAS', 'PAPAS FRITAS', 'COCOA', 'TE', 'LAVAVAJILLAS',
       'LIMPIADOR MULTIUSOS', 'DETERGENTE LIQUIDO', 'GELATINA',
       'CONSERVA DE FRUTA', 'DETERGENTE EN POLVO', 'AVENA',
       'PANIFICACION', 'MAZAMORRA', 'MODIFICADORES LACTEOS', 'CERVEZA',
       'TABLETA TAZA'], dtype=object)

In [88]:
df_ventas.desc_marca.unique()

array(['CIELO', 'ORO', 'PULP', 'VIDA', 'BIG', 'KR', 'SPORADE', 'VOLT',
       'CIFRUT', 'D GUSSTO', 'DEST', 'MUCHO GUSSTO', 'BIO ALOE',
       'PAPI RICAS', 'FREE TEA', 'CIELO ALCALINA', 'BIO AMAYU', 'FUSION',
       'BLING', 'TRES CRUCES'], dtype=object)

In [89]:
for giro_v in df_ventas["desc_subgiro"].unique():
    temp = df_ventas[(df_ventas["desc_subgiro"] == giro_v)]
    ranks = temp.groupby("desc_categoria")["cant_cajafisicavta"].sum().reset_index()
    ranks.columns = ["index","desc_categoria"]
    ranks = ranks.sort_values(by="desc_categoria", ascending=False)
    if len(list(ranks["desc_categoria"])) <= 5:
        ranks["Ranking"] = range(1, len(ranks) + 1)
    elif len(list(ranks["desc_categoria"])) > 5:
        a = list(ranks["desc_categoria"])
        b = [1, 1, 2, 2]
        # Calculamos el multiplicador para el mapeo de pesos segun la varianza
        if np.std(a) / np.mean(a) <= 1.2:
            multiplicador = 4
        else:
            multiplicador = 2
        if len(a) > 5:
            for i in range(4, len(a)):
                if a[3] <= a[i] * multiplicador:
                    b.append(3)
                else:
                    b.append(3 + i)
        ranks["Ranking"] = b

    print("*" * 20)
    print("GIRO: ", giro_v)
    print("Categorias: ", list(ranks["index"]))
    print("Counts: ", list(ranks["desc_categoria"]))
    print(ranks.set_index("index")["Ranking"].to_dict())
    mapeo_diccionario[giro_v] = ranks.set_index("index")["Ranking"].to_dict()

********************
GIRO:  BODEGAS, TIENDAS
Categorias:  ['AGUA', 'GASEOSAS', 'BEBIDAS DEPORTIVAS', 'JUGOS LIGEROS', 'ENERGIZANTE', 'NECTAR', 'PAPAS FRITAS', 'CERVEZA', 'DETERGENTE EN POLVO', 'TE', 'CAFE', 'CONSERVA DE PESCADO', 'ACEITE', 'SUAVIZANTES', 'COCOA', 'PANIFICACION', 'LAVAVAJILLAS', 'DETERGENTE LIQUIDO', 'CONSERVA DE FRUTA', 'LIMPIADOR MULTIUSOS', 'PASTAS', 'GELATINA', 'AVENA', 'MAZAMORRA', 'MODIFICADORES LACTEOS', 'TABLETA TAZA']
Counts:  [3094120.222222222, 2344876.2333333334, 1393890.6666666667, 612817.6666666666, 608623.5833333334, 340535.8333333333, 169193.16666666666, 130710.0, 110298.52380952382, 72138.0, 58176.0, 53659.666666666664, 40486.333333333336, 39276.0, 35120.0, 24397.0, 18854.666666666668, 14110.0, 13298.0, 12228.0, 8398.0, 7217.0, 1512.0, 1500.0, 1077.0, 112.0]
{'AGUA': 1, 'GASEOSAS': 1, 'BEBIDAS DEPORTIVAS': 2, 'JUGOS LIGEROS': 2, 'ENERGIZANTE': 3, 'NECTAR': 3, 'PAPAS FRITAS': 9, 'CERVEZA': 10, 'DETERGENTE EN POLVO': 11, 'TE': 12, 'CAFE': 13, 'CONSERVA DE

## 3.2. Filtro de SKU por Ruta
Cada Ruta tiene una cantidad de SKUs distintos, la recomendacion será distinta para cada ruta, sin embargo, aquellas rutas que tengan pocos SKUs serán tratadas como una sola ruta

In [90]:
rutas = (
    df_ventas.groupby(["cod_ruta"])["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()["cod_ruta"]
    .unique()
)
rutas

array([14451, 14452, 12970, 10029, 14121, 14508, 14523, 14602, 10602,
       10040, 14504, 14106, 14500, 14604, 10044, 11617, 12961, 14118,
       12963, 14505, 14502, 12972, 12958, 14109, 14105, 14506, 10034,
       14456, 14116, 14104, 10027, 10037, 12960, 14510, 14450, 14503,
       10046, 13817, 10031, 10030, 14509, 12975, 13591, 14120, 14454,
       14115, 10650, 14609, 10004, 12969, 10035, 10007, 10032, 14112,
       14100, 10018, 10045, 10644, 14501, 10017, 10014, 14103, 10048,
       14459, 10024, 14110, 11616, 10057, 10043, 10648, 10064, 10052,
       14111, 14102, 14455, 14117, 10021, 14603, 10640, 12952, 12962,
       14108, 10051, 10050, 14521, 10016, 10056, 12973, 10062, 10060,
       14605, 13803, 14507, 10449, 11600, 13577, 10005, 11611, 10058,
       10059, 12951, 10054, 13811, 12500, 14458, 10038, 10002, 10647,
       10008, 12953, 10445, 10615, 12955, 14457, 14101, 13850, 14606,
       10042, 10033, 14461, 10001, 10019, 14119, 10653, 10023, 10067,
       10028, 13815,

In [91]:
low_sku_ruta = []

**Para rutas que tengan más de 55 SKUS**

In [92]:
for ruta in rutas:
    print("*" * 21)
    print("Ruta:", ruta)
    temp = df_ventas[(df_ventas["cod_ruta"] == ruta)]
    print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
    # print(f"Giros en ruta {ruta}:")
    # print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
    if temp["cod_articulo_magic"].nunique() < 5: # Era 55 antes, pero mejor no combinar rutas
        low_sku_ruta.append(ruta)
    else:
        temp.to_csv(f"Processed/rutas_peru/D_{ruta}_ventas.csv", index=False)

*********************
Ruta: 14451
SKUs disponibles: 153
*********************
Ruta: 14452
SKUs disponibles: 156
*********************
Ruta: 12970
SKUs disponibles: 140
*********************
Ruta: 10029
SKUs disponibles: 146
*********************
Ruta: 14121
SKUs disponibles: 115
*********************
Ruta: 14508
SKUs disponibles: 151
*********************
Ruta: 14523
SKUs disponibles: 149
*********************
Ruta: 14602
SKUs disponibles: 137
*********************
Ruta: 10602
SKUs disponibles: 125
*********************
Ruta: 10040
SKUs disponibles: 144
*********************
Ruta: 14504
SKUs disponibles: 142
*********************
Ruta: 14106
SKUs disponibles: 99
*********************
Ruta: 14500
SKUs disponibles: 139
*********************
Ruta: 14604
SKUs disponibles: 151
*********************
Ruta: 10044
SKUs disponibles: 150
*********************
Ruta: 11617
SKUs disponibles: 112
*********************
Ruta: 12961
SKUs disponibles: 150
*********************
Ruta: 14118
SKUs disponible

**Para rutas que tengan menos de 45 SKUS**

In [93]:
print("*" * 21)
print("Rutas:", low_sku_ruta)
temp = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
# print(f"Giros en ruta {ruta}:")
# print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
temp.to_csv("Processed/rutas_peru/D_low_ruta_ventas.csv", index=False)

*********************
Rutas: []
SKUs disponibles: 0


# 4. Modelo Pedido Sugerido

## 4.1. Importando librerias necesarias

In [94]:
import os

from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType
from pyspark.sql.functions import (
    when,
    col,
    regexp_replace,
    concat,
    countDistinct,
    lit,
    monotonically_increasing_id,
    hash,
)

from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [95]:
!java -version

openjdk version "11.0.30" 2026-01-20 LTS
OpenJDK Runtime Environment Corretto-11.0.30.7.1 (build 11.0.30+7-LTS)
OpenJDK 64-Bit Server VM Corretto-11.0.30.7.1 (build 11.0.30+7-LTS, mixed mode)


In [96]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

## 4.2 Construccion del algoritmo ALS

In [97]:
def als_training_job(cedi, sku_len):

    spark = (
        SparkSession.builder.config("spark.memory.offHeap.enabled", "true")
        .config("spark.memory.offHeap.size", "10G")
        .appName("K-Means for ALS")
        .getOrCreate()
    )

    ventas = (
        spark.read.format("csv")
        .options(header=True, delimiter=",")
        .load(f"Processed/rutas_peru/D_{cedi}_ventas.csv")
    )

    ventas = ventas.na.drop(subset=["fecha_liquidacion"])
    ventas = ventas.groupBy(
        ["id_cliente", "cod_articulo_magic", "cod_compania", "cod_cliente"]
    ).agg(countDistinct("fecha_liquidacion"))
    ventas = ventas.select(
        col("id_cliente").alias("id_cliente"),
        col("count(fecha_liquidacion)").alias("frecuencia"),
        col("cod_articulo_magic").alias("cod_articulo_magic"),
        col("cod_compania").alias("cod_compania"),
        col("cod_cliente").alias("cod_cliente"),
    )

    input_cols = ["frecuencia"]
    vec_assembler = VectorAssembler(inputCols=input_cols, outputCol="features")
    ventas_kmeans = vec_assembler.transform(ventas)
    ventas_kmeans = ventas_kmeans.fillna(0.0, subset=["frecuencia"])

    kmeans = KMeans(featuresCol="features", k=5)
    model = kmeans.fit(ventas_kmeans)

    resumen_ventas = model.transform(ventas_kmeans).groupby("prediction").max()
    resumen_ventas.orderBy(["max(frecuencia)"], ascending=0)
    resumen_ventas = resumen_ventas.select(
        col("prediction").alias("prediction"), col("max(frecuencia)").alias("max")
    )

    maxes = resumen_ventas.select("max").toPandas()["max"]
    maxes = maxes.astype(float)
    maxes = list(maxes)

    labels = [5, 4, 3, 2, 1]
    rank_map = dict(zip(labels, maxes))

    ventas = ventas.withColumn("frecuencia", col("frecuencia").cast(DoubleType()))
    ventas = ventas.withColumn(
        "rating",
        when(ventas["frecuencia"] < rank_map.get(labels[-1]), labels[-1])
        .when(ventas["frecuencia"] < rank_map.get(labels[-2]), labels[-2])
        .when(ventas["frecuencia"] < rank_map.get(labels[-3]), labels[-3])
        .when(ventas["frecuencia"] < rank_map.get(labels[-4]), labels[-4])
        .otherwise(labels[-5]),
    )

    cols = ["id_cliente", "cod_compania", "cod_cliente", "cod_articulo_magic", "rating"]
    als_records = ventas.select(*cols)

    # als_records = als_records.withColumn("clienteId",concat("cod_compania","cod_cliente"))
    als_records = als_records.withColumn(
        "clienteId", concat(col("cod_compania"), lit("|"), col("cod_cliente"))
    )
    # als_records = als_records.withColumn("clienteId", col("clienteId").cast(IntegerType()))
    # Crear una columna numérica para usar en el modelo ALS
    # als_records = als_records.withColumn("clienteId_numeric", monotonically_increasing_id())
    als_records = als_records.withColumn("clienteId_numeric", hash(col("clienteId")))
    als_records = als_records.withColumn(
        "cod_articulo_magic", col("cod_articulo_magic").cast(IntegerType())
    )

    als_records = als_records.na.drop(
        subset=["clienteId_numeric", "cod_articulo_magic"]
    )

    input_cols = ["clienteId", "clienteId_numeric", "cod_articulo_magic", "rating"]
    als_records = als_records.select(*input_cols)

    als_records = als_records.dropDuplicates(["clienteId", "cod_articulo_magic"])

    als = ALS(
        rank=10,
        maxIter=5,
        implicitPrefs=True,
        ratingCol="rating",
        itemCol="cod_articulo_magic",
        userCol="clienteId_numeric",
    )
    model = als.fit(als_records)

    # obtener matrices de descomposicion (usuario y articulos)
    user_matrix = model.userFactors
    item_matrix = model.itemFactors

    user_matrix_pd = user_matrix.toPandas()
    item_matrix_pd = item_matrix.toPandas()

    recs = model.recommendForAllUsers(sku_len)
    # recs = recs.select("ClienteId","recommendations.cod_articulo_magic")
    recs = recs.select("clienteId_numeric", "recommendations.cod_articulo_magic")
    # Hacer un join con als_records para recuperar la columna 'clienteId' original
    recs = recs.join(
        als_records.select("clienteId", "clienteId_numeric"),
        on="clienteId_numeric",
        how="left",
    )

    # Convertir a pandas
    recs_to_parse = recs.select("clienteId", "cod_articulo_magic").toPandas()
    # recs_to_parse = recs.toPandas()

    lista_rec = [f"r{i+1}" for i in range(sku_len)]

    # recs_to_parse[lista_rec] = pd.DataFrame(recs_to_parse['cod_articulo_magic'].tolist(), index= recs_to_parse.index)
    new_cols = pd.DataFrame(
        recs_to_parse["cod_articulo_magic"].tolist(),
        index=recs_to_parse.index,
        columns=lista_rec,
    )
    recs_to_parse = pd.concat([recs_to_parse, new_cols], axis=1)

    client_recs = pd.melt(
        recs_to_parse, id_vars=["clienteId"], value_vars=lista_rec
    )

    # client_recs['compania'] = client_recs["clienteId"].apply(lambda x: str(x)[:2])
    client_recs["compania"] = client_recs["clienteId"].str.split("|").str[-2]
    # client_recs['cliente'] = client_recs["clienteId"].apply(lambda x: str(x)[2:])
    client_recs["cliente"] = client_recs["clienteId"].str.split("|").str[-1]
    client_recs["compania"] = client_recs["compania"].apply(lambda x: x.rjust(4, "0"))
    client_recs["id_cliente"] = client_recs["compania"] + "|" + client_recs["cliente"]
    client_recs["cod_articulo_magic"] = client_recs["value"]

    client_recs = (
        client_recs[["id_cliente", "cod_articulo_magic"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Obtener el nombre del archivo CSV
    nombre_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo ya existe
    try:
        # Intentar abrir el archivo para lectura
        with open(nombre_archivo, "r") as f:
            # Si el archivo existe, header=False para no escribir cabeceras
            client_recs.to_csv(
                nombre_archivo, mode="a", header=False, index=False, sep=";"
            )
    except FileNotFoundError:
        # Si el archivo no existe, header=True para escribir las cabeceras
        client_recs.to_csv(nombre_archivo, mode="a", header=True, index=False, sep=";")

## 4.3 Corriendo PS

In [98]:
def runALS():
    # Ruta del archivo que deseas verificar y eliminar si existe
    ruta_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo existe
    if os.path.exists(ruta_archivo):
        # Eliminar el archivo si existe
        os.remove(ruta_archivo)
        print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
    else:
        print(f"El archivo en la ruta {ruta_archivo} no existe.")
    # CORRIENDO ALS PARA RUTAS DE MAS DE 45 SKUs
    for ruta in rutas:
        print("*" * 21)
        temp = pd.read_csv(f"Processed/rutas_peru/D_{ruta}_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en ruta {ruta}:", sku_len)
        als_training_job(ruta, sku_len)
    # CORRIENDO ALS PARA rutas que tengan menos de 45 SKUS
    if len(low_sku_ruta) != 0:
        temp = pd.read_csv(f"Processed/rutas_peru/D_low_ruta_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en low ruta {low_sku_ruta}:", sku_len)
        als_training_job("low_ruta", sku_len)
    else:
        print("No existen rutas con pocos skus :D")

In [99]:
rutas = rutas[~pd.Series(rutas).isin(pd.Series(low_sku_ruta))]
rutas

array([14451, 14452, 12970, 10029, 14121, 14508, 14523, 14602, 10602,
       10040, 14504, 14106, 14500, 14604, 10044, 11617, 12961, 14118,
       12963, 14505, 14502, 12972, 12958, 14109, 14105, 14506, 10034,
       14456, 14116, 14104, 10027, 10037, 12960, 14510, 14450, 14503,
       10046, 13817, 10031, 10030, 14509, 12975, 13591, 14120, 14454,
       14115, 10650, 14609, 10004, 12969, 10035, 10007, 10032, 14112,
       14100, 10018, 10045, 10644, 14501, 10017, 10014, 14103, 10048,
       14459, 10024, 14110, 11616, 10057, 10043, 10648, 10064, 10052,
       14111, 14102, 14455, 14117, 10021, 14603, 10640, 12952, 12962,
       14108, 10051, 10050, 14521, 10016, 10056, 12973, 10062, 10060,
       14605, 13803, 14507, 10449, 11600, 13577, 10005, 11611, 10058,
       10059, 12951, 10054, 13811, 12500, 14458, 10038, 10002, 10647,
       10008, 12953, 10445, 10615, 12955, 14457, 14101, 13850, 14606,
       10042, 10033, 14461, 10001, 10019, 14119, 10653, 10023, 10067,
       10028, 13815,

In [ ]:
%%time
runALS()

El archivo en la ruta Output/D_rutas_rec.csv ha sido eliminado.
*********************
SKUs disponibles en ruta 14451: 153


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/31 20:32:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/31 20:33:03 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/31 20:33:03 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
26/03/31 20:33:03 WARN InstanceBuilder$NativeLAPACK: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


*********************
SKUs disponibles en ruta 14452: 156
*********************
SKUs disponibles en ruta 12970: 140
*********************
SKUs disponibles en ruta 10029: 146
*********************
SKUs disponibles en ruta 14121: 115
*********************
SKUs disponibles en ruta 14508: 151
*********************
SKUs disponibles en ruta 14523: 149
*********************
SKUs disponibles en ruta 14602: 137
*********************
SKUs disponibles en ruta 10602: 125


# 5. Ajustes al output del algoritmo ALS

## 5.-9 Quedarnos con skus que hayan tenido ventas en los ultimos 30 dias

In [ ]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
pan_rec["id_cliente"] = "PE|" + pan_rec["id_cliente"]

In [ ]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [ ]:
pan_rec.shape

In [ ]:
pan_rec.id_cliente.nunique()

In [ ]:
df_ventas = pd.read_parquet("Processed/peru_ventas_manana.parquet")

In [ ]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(df_ventas["fecha_liquidacion"])

In [ ]:
# Definir el umbral de tiempo (últimos 30 días desde hoy)
# bajamos de 30 a 14 dias 2025-07-25
hoy = pd.Timestamp.today()
fecha_limite = hoy - pd.Timedelta(days=14)
print(fecha_limite)
# Filtrar ventas en los últimos 30 días
ventas_filtradas = df_ventas[df_ventas["fecha_liquidacion"] >= fecha_limite]

In [ ]:
# Obtener los productos vendidos en cada ruta
productos_por_ruta = ventas_filtradas.groupby("cod_ruta")["cod_articulo_magic"].unique().reset_index()

In [ ]:
# Unir con los productos vendidos en cada ruta
recomendaciones_validas = pan_rec.merge(productos_por_ruta, on="cod_ruta", how="inner")

In [ ]:
recomendaciones_validas.columns = ['id_cliente', 'cod_articulo_magic', 'cod_ruta','lista_sku_ruta']

In [ ]:
recomendaciones_validas.shape

In [ ]:
recomendaciones_validas  = recomendaciones_validas[recomendaciones_validas.apply(lambda row: row["cod_articulo_magic"] in row["lista_sku_ruta"], axis=1)].reset_index(drop=True)

In [ ]:
recomendaciones_validas.shape

In [ ]:
pan_rec = recomendaciones_validas[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [ ]:
pan_rec.head()

In [ ]:
pan_rec.shape

In [ ]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-8 Recomendar de acuerdo a Subida, Bajada, Mantener

In [ ]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
#pan_rec["id_cliente"] = "PE|" + pan_rec["id_cliente"]

In [ ]:
pan_rec.shape

In [ ]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [ ]:
def desplazar_nulos_fila(fila):
    return pd.Series(sorted(fila, key=pd.isna), index=fila.index)

In [ ]:
# Definir una función para asignar los valores subida, mantener y bajar
def clasificar_valor(x):
    if x > 0:
        return "S"
    elif x == 0:
        return "M"
    else:
        return "B"

In [ ]:
# df_ventas['fecha_liquidacion'] = pd.to_datetime(df_ventas['fecha_liquidacion'])

# Definir la fecha de referencia (hoy)
fecha_actual = datetime.now(pytz.timezone("America/Lima"))#.strftime('%Y-%m-%d')
fecha_30dias_atras = (fecha_actual - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
fecha_60dias_atras = (fecha_actual - pd.Timedelta(days=60)).strftime('%Y-%m-%d')

print("fecha_actual",fecha_actual)
print("fecha_30dias_atras",fecha_30dias_atras)
print("fecha_60dias_atras",fecha_60dias_atras)

In [ ]:
# Filtrar los últimos 30 días
df_ultimos_30 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_30dias_atras) 
                          & (df_ventas['fecha_liquidacion'] <= fecha_actual.strftime('%Y-%m-%d'))]

# Filtrar los días 31 a 60
df_31_60 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_60dias_atras)
              & (df_ventas['fecha_liquidacion'] <= fecha_30dias_atras)]

# Agrupar por codigo_usuario, ruta, codigo_producto y sumar ventas
ventas_ultimos_30 = df_ultimos_30.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_ultimos_30["mes"] = "0_30"
ventas_31_60 = df_31_60.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_31_60["mes"] = "31_60"

df_grouped = pd.concat([ventas_ultimos_30,ventas_31_60],axis=0,ignore_index=True)
df_grouped = pd.pivot_table(
    df_grouped,
    values="imp_netovta",
    index=["cod_ruta", "cod_articulo_magic"],
    columns=["mes"],
    aggfunc="sum",
).reset_index()

# Rellenando con 0 si es nulo
df_grouped["0_30"] = df_grouped["0_30"].fillna(0)
df_grouped["31_60"] = df_grouped["31_60"].fillna(0)

# Asignando variacion de productos por mes
df_grouped["v1_2"] = (
    (df_grouped[df_grouped.columns[2]] - df_grouped[df_grouped.columns[3]])
    / df_grouped[df_grouped.columns[3]]
    * 100
)
df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)
df_grouped["v1_2"] = df_grouped["v1_2"].fillna(-1)
df_grouped["vp"] = (df_grouped["v1_2"])

In [ ]:
# Aplicar la función a la columna 'valor' y guardar el resultado en una nueva columna 'flag'
df_grouped["flag"] = df_grouped["vp"].apply(lambda x: clasificar_valor(x))
# Definir el mapeo de valores
mapeo_flag = {"S": 0, "M": 1, "B": 2}
# Aplicar el mapeo a la columna 'clasificacion' y guardar el resultado en una nueva columna 'flag_rank'
df_grouped["flag_rank"] = df_grouped["flag"].map(mapeo_flag)

In [ ]:
df_grouped

In [ ]:
pan_rec.shape

In [ ]:
pan_rec = pd.merge(pan_rec,df_grouped[["cod_ruta", "cod_articulo_magic","flag_rank"]],on=["cod_ruta", "cod_articulo_magic"],how="left")

In [ ]:
pan_rec["flag_rank"] = pan_rec["flag_rank"].fillna(3)

In [ ]:
# Agregar una columna con el orden original
pan_rec["original_order"] = pan_rec.index

In [ ]:
# Ordenar por cod_cliente, rank_producto (ascendente) y mantener el orden original en caso de empate
pan_rec = pan_rec.sort_values(by=["id_cliente", "flag_rank", "original_order"], ascending=[True, True, True])#.drop(columns=["original_order"])

In [ ]:
pan_rec.head()

In [ ]:
pan_rec.shape

In [ ]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [ ]:
pan_rec.head()

In [ ]:
pan_rec.shape

In [ ]:
pan_rec.id_cliente.nunique()

In [ ]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-7 Usamos el archivo de validacion para no recomendar SKUs que no se deben a las rutas

In [ ]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "PE|" + pan_rec["id_cliente"]

In [ ]:
pan_rec.shape

In [ ]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [ ]:
pan_rec.isnull().sum()

In [ ]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

In [ ]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

In [ ]:
# Ruta del archivo en S3
s3_path = "s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/peru/maestro_productos_peru000"

# Leer el CSV directamente a un DataFrame
skus_val = pd.read_csv(s3_path,sep = ";")
# skus_val = pd.read_csv("maestra_productos_peru_20250217.csv",sep = ";")

In [ ]:
# skus_val = skus_val[skus_val.cod_compania==70] # peru tiene varias compañias
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print("fecha_proceso: ", hoy)
# Filtrar solo las filas con la fecha de hoy
# skus_val = skus_val[skus_val['fecha_proceso'] == hoy]
skus_val["cod_compania"] = (skus_val["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
skus_val["id_cliente"] = (
    "PE"
    + "|"
    + skus_val["cod_compania"].astype(str)
    + "|"
    + skus_val["cod_cliente"].astype(str)
)

In [ ]:
skus_val.head()

In [ ]:
skus_val.shape

In [ ]:
# Cada ruta pertenece a una compañia
skus_val.groupby("cod_ruta").cod_compania.nunique().mean()

In [ ]:
pan_rec.shape

In [ ]:
pan_rec = pd.merge(pan_rec,skus_val[['cod_articulo_magic', 'id_cliente']].drop_duplicates(),left_on=["id_cliente","cod_articulo_magic"],right_on=["id_cliente","cod_articulo_magic"],how="inner")

In [ ]:
pan_rec.shape

In [ ]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

In [ ]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

In [ ]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]]

In [ ]:
pan_rec.head()

In [ ]:
pan_rec.id_cliente.nunique()

In [ ]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-5 Filtrar SKUs por STOCK

In [ ]:
stock = pd.read_csv("s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/peru/D_stock_pe.csv")

In [ ]:
stock.drop(columns=["Fecha","Database"], inplace=True)
stock.columns=["cod_compania","cod_sucursal","cod_articulo_magic","stock_cf"]

In [ ]:
stock["cod_compania"] = (stock["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
stock["cod_sucursal"] = (stock["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [ ]:
fecha_hace_12_dias = (datetime.today() - timedelta(days=12)).strftime('%Y-%m-%d')
print(fecha_hace_12_dias)

In [ ]:
# Promedio diario de cajas fisicas por sucursal y compañia de los ultimos 12 dias
prom_diario_vta = df_ventas[df_ventas.cant_cajafisicavta > 0]
prom_diario_vta = df_ventas[df_ventas.fecha_liquidacion>=fecha_hace_12_dias].groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic","fecha_liquidacion"]).cant_cajafisicavta.sum().reset_index().groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic"]).cant_cajafisicavta.mean().reset_index()
prom_diario_vta["cod_compania"] = (prom_diario_vta["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
prom_diario_vta["cod_sucursal"] = (prom_diario_vta["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [ ]:
df_stock = pd.merge(prom_diario_vta,stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"],how="left")
df_stock["dias_stock"] = df_stock["stock_cf"]/df_stock["cant_cajafisicavta"]
df_stock =  df_stock[(df_stock.dias_stock > 3) & (df_stock.cant_cajafisicavta > 0)].reset_index(drop=True)

In [ ]:
pan_rec.nunique()

In [ ]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

In [ ]:
pan_rec.shape

In [ ]:
# Cruzamos stock

In [ ]:
test = pd.merge(pan_rec,df_ventas[["id_cliente","cod_compania","cod_sucursal"]].drop_duplicates(),on="id_cliente",how="left")

In [ ]:
test["cod_compania"] = (test["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
test["cod_sucursal"] = (test["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [ ]:
test_f = pd.merge(test, df_stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"], how="inner")

In [ ]:
test_f.head()

In [ ]:
test_f.shape

In [ ]:
test_f.nunique()

In [ ]:
test_f.groupby("id_cliente").cod_articulo_magic.nunique().mean()

In [ ]:
pan_rec = test_f[["id_cliente","cod_articulo_magic"]].copy()

In [ ]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-4 Quitar SKUs en especifico de un XLSX

In [ ]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [ ]:
sku_con_precio = pd.read_excel(
    "LISTA SKUS - NN.xlsx", sheet_name="Hoja1"
)

In [ ]:
sku_con_precio

In [ ]:
# Original size de recomendacioness
pan_rec.shape

In [ ]:
pan_rec = pan_rec[
    (~pan_rec.cod_articulo_magic.isin(sku_con_precio["CÓDIGO"].unique()))
].reset_index(drop=True)

In [ ]:
# Size de recomendaciones luego de filtar solo a los del excel de SKUs activos
pan_rec.shape

In [ ]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-3 Quitar Recomendaciones de SKUS en especifico

In [ ]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [ ]:
# SKUS sin stock
skus_sin_precio = [622908, 622102, 699373, 622103, 622672, 699790, 622841, 623140, 699423, 622735, 622192, 622189, 623130, 623130, 699127, 622396,608462,623557,622838,621939,623566,622313
                  ,621994,622163,622391,622392,622393,622395,622630,699367,699892,622356,623854,699361,699366,699550,619038,623872,621766,621804,621766,621804,622163,622356,622387,622390,622391,622392,
                  622393,622394,622395,622396,622439,622783,622841,623130,
                  623150,623456,623753,698987,699346,699363,699367,699373,
                  699425,699493,699494,623873,623875,624052,699868, 699494,623873, 699927, 622439, 623875, 623753, 622313, 699495, 622387, 622393, 623087, 622838,623557,623565,624239,624245,624285, 622079
                  ]

In [ ]:
pan_rec=pd.merge(pan_rec, pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [ ]:
pan_rec=pan_rec[~((pan_rec.cod_ruta.isin([14608,14450, 14471,12967, 12968, 12958, 12972]) & (pan_rec.cod_articulo_magic == 608597)))]

In [ ]:
pan_rec = pan_rec[~(pan_rec["cod_articulo_magic"].isin(skus_sin_precio))].reset_index(
    drop=True
)

In [ ]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-2 Quitar Recomendaciones de las ultimas 2 SEMANAS (para eliminar recomendaciones pasadas -> SOLO USAR CUANDO HAYA HISTORIA DE RECOMENDACIONES)

In [ ]:
import os
import re

In [ ]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [ ]:
master_prod = pd.read_csv("Input/PE_maestro_productos.csv")

In [ ]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

In [ ]:
ruta_archivo = f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv"
# Verificar si el archivo existe
if os.path.exists(ruta_archivo):
    # Eliminar el archivo si existe
    os.remove(ruta_archivo)
    print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
else:
    print(f"El archivo en la ruta {ruta_archivo} no existe.")

In [ ]:
# Directorio donde se encuentran los archivos
directorio = "Output/PS_piloto_v1/"

# Expresión regular para encontrar fechas en el nombre de los archivos
patron_fecha = r"\d{4}-\d{2}-\d{2}"

# Lista para almacenar las fechas extraídas
fechas = []

# Iterar sobre los archivos en el directorio
for archivo in os.listdir(directorio):
    # Verificar si el archivo es un archivo CSV y coincide con el patrón de nombres
    if archivo.endswith(".csv") and re.match(
        r"^D_base_pedidos_\d{4}-\d{2}-\d{2}\.csv$", archivo
    ):
        # Extraer la fecha del nombre del archivo
        fecha = re.search(patron_fecha, archivo).group()
        # Agregar la fecha a la lista
        fechas.append(fecha)

In [ ]:
last_7_days_2 = sorted(fechas)[-14:]
last_7_days_2

In [ ]:
# Leer recomendaciones de los ultimos 7 dias y cruzar con clientes que tienen recomendacion para mañana
# CAMBIAR RUTA SI ES NECESARIO
last_7_days_recs = pd.DataFrame()
for fecha_rec in last_7_days_2:
    df_temp = pd.read_csv(
        f"Output/PS_piloto_v1/D_base_pedidos_{fecha_rec}.csv",
        dtype={"Compania": "str", "Cliente": "str"},
    )
    df_temp["id_cliente"] = "PE|" + df_temp["Compania"] + "|" + df_temp["Cliente"]
    df_temp = df_temp[df_temp["id_cliente"].isin(pan_rec["id_cliente"].unique())]
    last_7_days_recs = pd.concat([last_7_days_recs, df_temp], axis=0)
    print(f"{fecha_rec} done")

In [ ]:
# Utilizamos merge para combinar los dataframes, utilizando la columna 'Cliente' como clave
df_combinado = pd.merge(
    pan_rec,
    last_7_days_recs,
    left_on=["id_cliente", "cod_articulo_magic"],
    right_on=["id_cliente", "Producto"],
    how="left",
    indicator=True,
)
# Filtramos los registros que solo están en el DataFrame 1
df_resultado = df_combinado[df_combinado["_merge"] == "left_only"][
    ["id_cliente", "cod_articulo_magic"]
].reset_index(drop=True)
df_resultado

In [ ]:
pan_rec.shape

In [ ]:
pan_rec.id_cliente.nunique()

In [ ]:
pan_rec.cod_articulo_magic.nunique()

In [ ]:
df_resultado.shape

In [ ]:
df_resultado.id_cliente.nunique()

In [ ]:
df_resultado.cod_articulo_magic.nunique()

In [ ]:
df_resultado

In [ ]:
df_resultado.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.0 Calcular Irregularidad de clientes

In [ ]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [ ]:
pan_rec["id_cliente"].nunique()

In [ ]:
now = pd.to_datetime(datetime.now(pytz.timezone("America/Lima")).strftime("%Y-%m-01"))
fecha_doce_meses_atras = now - pd.DateOffset(months=12)
lista_m12 = [fecha_doce_meses_atras + pd.DateOffset(months=i) for i in range(12)]
lista_m12

fecha_seis_meses_atras = now - pd.DateOffset(months=6)
lista_m6 = [fecha_seis_meses_atras + pd.DateOffset(months=i) for i in range(6)]
lista_m6

In [ ]:
temp_df_ventas = df_ventas[df_ventas["id_cliente"].isin(pan_rec["id_cliente"].unique())]

In [ ]:
qw = (
    temp_df_ventas[["id_cliente", "mes"]]
    .drop_duplicates()
    .sort_values(["id_cliente", "mes"])
    .groupby("id_cliente")
    .tail(12)
    .reset_index(drop=True)
)
qw["mes"] = pd.to_datetime(qw["mes"])
qw["m12"] = qw["mes"].isin(lista_m12)
qw["m6"] = qw["mes"].isin(lista_m6)

In [ ]:
categoria_cliente = qw.groupby("id_cliente")[["m12", "m6"]].sum().reset_index()

In [ ]:
# Definir condiciones
condicion_super = categoria_cliente["m12"] == 12
condicion_frecuente = (
    (categoria_cliente["m12"] < 12)
    & (categoria_cliente["m12"] >= 6)
    & (categoria_cliente["m6"] == 6)
)
condicion_regulares = (categoria_cliente["m6"] <= 6) & (categoria_cliente["m12"] <= 6)
condicion_riesgo = (categoria_cliente["m6"] < 6) & (categoria_cliente["m6"] >= 4)
condicion_irregulares = categoria_cliente["m6"] < 4

# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_super, "categoria_cliente"] = "Super"
categoria_cliente.loc[condicion_frecuente, "categoria_cliente"] = "Frecuente"
categoria_cliente.loc[condicion_regulares, "categoria_cliente"] = "Regular"
categoria_cliente.loc[condicion_riesgo, "categoria_cliente"] = "Riesgo"
categoria_cliente.loc[condicion_irregulares, "categoria_cliente"] = "Irregular"

**Recalcular Irregularidad**

In [ ]:
# Definir condiciones
condicion_regular = (
    (categoria_cliente["categoria_cliente"] == "Super")
    | (categoria_cliente["categoria_cliente"] == "Frecuente")
    | (categoria_cliente["categoria_cliente"] == "Regular")
    | (categoria_cliente["categoria_cliente"] == "Riesgo")
)
condicion_irregular = categoria_cliente["categoria_cliente"] == "Irregular"
# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_regular, "categoria_cliente_2"] = "Regular"
categoria_cliente.loc[condicion_irregular, "categoria_cliente_2"] = "Irregular"

In [ ]:
categoria_cliente

In [ ]:
categoria_cliente["categoria_cliente"].value_counts(dropna=False)

In [ ]:
categoria_cliente["categoria_cliente_2"].value_counts(dropna=False)

## 5.1. Obteniendo marcas de recomendaciones por cliente

In [ ]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [ ]:
pan_ventas = df_ventas[(df_ventas["cod_ruta"].isin(rutas))]
pan_ventas2 = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
pan_ventas = pd.concat([pan_ventas, pan_ventas2], axis=0)

In [ ]:
# Solo se usó para obtener informacion EXTRA
# pan_evo_venta=pd.read_csv("D_pan_EvolucionCompra.csv")

In [ ]:
# %%time
# def desplazar_nulos_fila(fila):
#     return pd.Series(sorted(fila, key=pd.isna), index=fila.index)

# #Convertir las Ventas de Nuevos Productos a 0 para que luego no sea afectada al desplazar los nulos
# for i in range (int((pan_evo_venta.shape[1]-4)/5)):
#     pan_evo_venta.loc[((~pan_evo_venta[f"mes_{i+1}_ventaBeb"].isnull())&(pan_evo_venta[f"mes_{i+1}_ventaNP"].isnull())),f"mes_{i+1}_ventaNP"]=0
#     pan_evo_venta.loc[((pan_evo_venta[f"mes_{i+1}_ventaBeb"].isnull())&(~pan_evo_venta[f"mes_{i+1}_ventaNP"].isnull())),f"mes_{i+1}_ventaBeb"]=0

# #Desplazar los meses nulos a la derecha
# pan_evo_venta=pan_evo_venta.apply(desplazar_nulos_fila, axis=1)
# pan_evo_venta

In [ ]:
marca_articulo = pan_ventas[["desc_categoria", "cod_articulo_magic"]].drop_duplicates()
print(marca_articulo.shape)
marca_articulo.head()

In [ ]:
cliente_rec_marca = pd.merge(
    pan_rec, marca_articulo, on="cod_articulo_magic", how="left"
)
cliente_rec_marca["desc_categoria"] = cliente_rec_marca["desc_categoria"].str.strip()
print(cliente_rec_marca.shape)
cliente_rec_marca

## 5.2. Calcular marcas distintas recomendadas por cliente

In [ ]:
cods2 = cliente_rec_marca.groupby("id_cliente")["desc_categoria"].nunique().reset_index()
print(cods2.shape)
cods2.head()

## 5.3 Quitando los SKUs de las ultimas 2 semanas (Evitar recompra)

In [ ]:
last_2_weeks = pd.to_datetime(datetime.now(pytz.timezone("America/Lima"))) - pd.DateOffset(
    days=14
)
last_2_weeks

In [ ]:
# Asegurarse de que 'fecha_liquidacion' esté en formato datetime
pan_ventas["fecha_liquidacion"] = pd.to_datetime(pan_ventas["fecha_liquidacion"])
# Convertir 'fecha_liquidacion' a la misma zona horaria que 'last_year'
pan_ventas["fecha_liquidacion"] = pan_ventas["fecha_liquidacion"].dt.tz_localize(
    "America/Lima", nonexistent="NaT", ambiguous="NaT"
)

In [ ]:
df_quitar_2_weeks = pan_ventas[pan_ventas["fecha_liquidacion"]>=last_2_weeks][["id_cliente","cod_articulo_magic"]].drop_duplicates().reset_index(drop=True)

In [ ]:
# Hacemos un merge para identificar coincidencias
cliente_rec_sin4 = cliente_rec_marca.merge(df_quitar_2_weeks, on=['id_cliente', 'cod_articulo_magic'], how='left', indicator=True)
# Filtramos solo los registros que NO están en df_quitar_2_weeks
cliente_rec_sin4 = cliente_rec_sin4[cliente_rec_sin4['_merge'] == 'left_only'].drop(columns=['_merge'])

In [ ]:
cliente_rec_marca.id_cliente.nunique()

In [ ]:
cliente_rec_sin4.id_cliente.nunique()

In [ ]:
cliente_rec_sin4

## 5.4 Filtro para calcula antiguedad de clientes

In [ ]:
pan_ventas = pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"})

In [ ]:
# Convertir la columna 'fecha_creacion' a tipo fecha
pan_ventas["fecha_creacion"] = pd.to_datetime(
    pan_ventas["fecha_creacion"], format="%Y%m%d"
)

In [ ]:
# Definir una función para etiquetar los clientes según su fecha de creación
def etiquetar_cliente(fecha_creacion):
    if pd.isnull(fecha_creacion):
        return (
            "nf"  # Si la fecha de creación es nula, etiquetar como 'nf' (no encontrada)
        )
    else:
        hoy = datetime.now()  # Obtener la fecha actual
        hace_12_meses = hoy - timedelta(days=365)  # Calcular hace 12 meses
        if fecha_creacion >= hace_12_meses:
            return "new_client"  # Si la fecha de creación está dentro de los últimos 12 meses, etiquetar como 'new_client'
        else:
            return "old_client"  # Si la fecha de creación es anterior a los últimos 12 meses, etiquetar como 'old_client'

In [ ]:
# Aplicar la función a la columna 'fecha_creacion' para crear la nueva columna 'tipo_cliente'
pan_ventas["antiguedad"] = pan_ventas["fecha_creacion"].apply(etiquetar_cliente)

In [ ]:
pan_antiguedad_clientes = (
    pan_ventas[["id_cliente", "fecha_creacion", "antiguedad"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

## 5.4.2 Obteniendo datos necesarios para armar DF para subir a SalesForce

In [ ]:
datos_para_salesforce = (
    pan_ventas.groupby(["id_cliente", "cod_compania", "cod_sucursal", "cod_cliente", "cod_ruta"])[
        ["id_cliente", "cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "cod_ruta"]
    ]
    .head(1)
    .reset_index(drop=True)
)

## 5.5 Juntando las Recomendaciones Finales
 - Ordenado de acuerdo al peso de la recomendación y el id de usuario.
 - Hay 5 skus de 5 marcas distintas que se recomiendan a cada usuario
 - Ningún SKU que haya comprado el usuario en sus ultimas X visitas será recomendado. La cantidad de visitas (X) a ignorar depende de sus visitas promedio mensuales.

In [ ]:
# Obtener primer SKU de cada marca de la recomendacion
final_rec = cliente_rec_sin4.groupby(["id_cliente", "desc_categoria"]).first().reset_index()
# Agregar Marca de sus primeras compras
final_rec = pd.merge(final_rec, cods2, how="left", on="id_cliente")
# Agregar descripcion de SKU recomendado
pan_prod = pan_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
pan_prod = pan_prod.groupby(["cod_articulo_magic"]).first().reset_index()
final_rec = pd.merge(final_rec, pan_prod, how="left", on="cod_articulo_magic")
# Cambiar nombre de columnas
final_rec.columns = ["id_cliente", "marca_rec", "sku", "len_marca_rec", "desc_rec"]
# Agregar Giro del cliente
giros = df_ventas[df_ventas["id_cliente"].isin(final_rec["id_cliente"].unique())][
    ["id_cliente", "desc_giro", "desc_subgiro"]
].drop_duplicates()
final_rec = pd.merge(final_rec, giros, on="id_cliente", how="left")
# Agregar si el cliente es Regular o Irregular
final_rec = pd.merge(
    final_rec,
    categoria_cliente[["id_cliente", "categoria_cliente_2"]],
    on="id_cliente",
    how="left",
)
final_rec.loc[final_rec["categoria_cliente_2"] == "Irregular", "desc_giro"] = np.nan
# Agregar antiguedad de cliente
final_rec = pd.merge(final_rec, pan_antiguedad_clientes, how="left", on="id_cliente")
# Agregar segmento de cliente
info_segmentos = pan_ventas[
    [
        "id_cliente",
        "new_segment",
        "dias_de_visita__c",
        "periodo_de_visita__c",
        "ultima_visita",
    ]
].drop_duplicates()
info_segmentos = info_segmentos.groupby(["id_cliente"]).first().reset_index()
final_rec = pd.merge(final_rec, info_segmentos, how="left", on="id_cliente")
# Agregar datos necesarios para SALESFORCE
final_rec = pd.merge(final_rec, datos_para_salesforce, how="left", on="id_cliente")
# Agregar Peso de marcas dependiendo del giro del cliente
final_rec["peso"] = final_rec.apply(
    lambda row: mapeo_diccionario.get(row["desc_subgiro"], {}).get(row["marca_rec"], 5),
    axis=1,
)

In [ ]:
# Ordenar la recomendacion de acuerdo al peso de la marca y nos quedamos con las 5 primeras marcas
final_rec = (
    final_rec.sort_values(["id_cliente", "peso"]).groupby(["id_cliente"]).head(5)
)
# Agregamos rank de la marca recomendada
final_rec["marca_rec_rank"] = final_rec.groupby("id_cliente").cumcount() + 1
final_rec = final_rec.reset_index(drop=True)

In [ ]:
final_rec["sku"] = final_rec["sku"].astype(int)
final_rec["peso"] = final_rec["peso"].astype(int)
final_rec["marca_rec_rank"] = final_rec["marca_rec_rank"].astype(int)

In [ ]:
final_rec

In [ ]:
final_rec.groupby("id_cliente")["sku"].count().mean()

In [ ]:
final_rec.groupby("id_cliente")["sku"].nunique().mean()

## 5.8. Guardando recomendacion para D&A y Comercial

In [ ]:
final_rec.id_cliente.nunique()

In [ ]:
final_rec.dias_de_visita__c.value_counts(dropna=False)

In [ ]:
final_rec.periodo_de_visita__c.value_counts(dropna=False)

In [ ]:
final_rec.ultima_visita.value_counts(dropna=False)

### 5.8.1 Para D&A

In [ ]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

In [ ]:
final_rec.to_csv(
    f"Output/PS_data_piloto_v1/D_pan_recs_data_{fecha_tomorrow}.csv", index=False
)

In [ ]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Peru/Output/PS_data_piloto_v1/D_pan_recs_data_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(final_rec, s3_path, index=False)

### PRIORIZAR SKUs

In [ ]:
# skus_a_priorizar = [623835, 623836, 623840]

In [ ]:
# skus_a_priorizar

In [ ]:
# skus_a_priorizar_p1 = pan_ventas[pan_ventas.cod_articulo_magic.isin(skus_a_priorizar)].groupby(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "cod_articulo_magic"]).cant_cajafisicavta.sum().reset_index().sort_values(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "cant_cajafisicavta"], ascending = [True,True,True,True,False]).reset_index(drop=True).groupby(
#     ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo"]).head(1).reset_index(drop=True)

In [ ]:
# skus_a_priorizar_p1 = skus_a_priorizar_p1.drop('cant_cajafisicavta', axis=1)

In [ ]:
# cli_faltantes = final_rec[~final_rec.cod_cliente.isin(skus_a_priorizar_p1.cod_cliente.astype(str).unique())].reset_index(drop=True)

In [ ]:
# skus_a_priorizar_p2 = pan_ventas[pan_ventas.id_cliente.isin(cli_faltantes.id_cliente.unique())][["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo"]].drop_duplicates().reset_index(drop=True)

In [ ]:
# # Para asignar un valor aleatorio a cada fila:
# skus_a_priorizar_p2['cod_articulo_magic'] = np.random.choice(skus_a_priorizar, size=len(skus_a_priorizar_p2))

In [ ]:
# skus_a_priorizar_final = pd.concat([skus_a_priorizar_p1, skus_a_priorizar_p2], axis=0, ignore_index=True)

In [ ]:
# skus_a_priorizar_final.rename(columns={'cod_articulo_magic': 'sku'}, inplace=True)

In [ ]:
# skus_a_priorizar_p1.cod_cliente.nunique()

In [ ]:
# skus_a_priorizar_p2.cod_cliente.nunique()

In [ ]:
# skus_a_priorizar_final.cod_cliente.nunique()

In [ ]:
# final_rec = pd.concat([skus_a_priorizar_final, final_rec], axis=0, ignore_index=True)

## 5.9. Filtro de Recomendaciones por Segmento de cliente

In [ ]:
# Definir una función para filtrar las filas según el valor de 'new_segment'
def filtrar_segmento(group):
    segmento = group["new_segment"].iloc[
        0
    ]  # Obtener el valor del segmento para el grupo
    if segmento == "BLINDAR":
        return group.head(1)  # Si el segmento es 'Blindar', mantener la primera fila 1
    elif segmento == "MANTENER":
        return group.head(
            2
        )  # Si el segmento es 'Mantener', mantener las dos primeras filas 2
    elif segmento == "DESARROLLAR":
        return group.head(
            3
        )  # Si el segmento es 'Desarrollar', mantener las tres primeras filas 3
    elif segmento == "OPTIMIZAR":
        return group.head(
            4
        )  # Si el segmento es 'Optimizar', mantener las cuatro primeras filas 4
    else:
        return group  # Si el segmento no es ninguno de los especificados, mantener todas las filas

In [ ]:
final_rec.nunique()

In [ ]:
final_rec.groupby("id_cliente").sku.nunique().mean()

In [ ]:
final_rec = (
    final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)
)

In [ ]:
final_rec.nunique()

In [ ]:
final_rec.groupby("id_cliente").sku.nunique().mean()

In [ ]:
final_rec["cod_compania"] = final_rec["cod_compania"].astype(int)
final_rec["cod_sucursal"] = final_rec["cod_sucursal"].astype(int)
final_rec["cod_cliente"] = final_rec["cod_cliente"].astype(int)
final_rec["cod_modulo"] = final_rec["cod_modulo"].astype(int)
final_rec["sku"] = final_rec["sku"].astype(int)

## 5.10. Aplicar formato para SF

In [ ]:
recomendaciones_para_salesforce = final_rec[
    ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "sku"]
]
recomendaciones_para_salesforce = recomendaciones_para_salesforce.drop_duplicates()
recomendaciones_para_salesforce["Pais"] = "PE"
recomendaciones_para_salesforce["Cajas"] = int(1)
recomendaciones_para_salesforce["Unidades"] = int(0)
recomendaciones_para_salesforce["Fecha"] = fecha_tomorrow
recomendaciones_para_salesforce = recomendaciones_para_salesforce[
    [
        "Pais",
        "cod_compania",
        "cod_sucursal",
        "cod_cliente",
        "cod_modulo",
        "sku",
        "Cajas",
        "Unidades",
        "Fecha",
    ]
]
recomendaciones_para_salesforce.columns = [
    "Pais",
    "Compania",
    "Sucursal",
    "Cliente",
    "Modulo",
    "Producto",
    "Cajas",
    "Unidades",
    "Fecha",
]

In [ ]:
recomendaciones_para_salesforce["Compania"] = recomendaciones_para_salesforce[
    "Compania"
].apply(lambda x: str(int(x)).rjust(4, "0"))
recomendaciones_para_salesforce["Sucursal"] = recomendaciones_para_salesforce[
    "Sucursal"
].apply(lambda x: str(int(x)).rjust(2, "0"))

In [ ]:
recomendaciones_para_salesforce.Sucursal.unique()

In [ ]:
recomendaciones_para_salesforce.Compania.unique()

In [ ]:
recomendaciones_para_salesforce.Cliente.nunique()

In [ ]:
recomendaciones_para_salesforce

In [ ]:
recomendaciones_para_salesforce.Cliente.nunique()

In [ ]:
rutas_cliente = df_ventas[["cod_cliente", "cod_ruta"]].drop_duplicates()

In [ ]:
recomendaciones_para_salesforce["Cliente"] = recomendaciones_para_salesforce["Cliente"].astype(str)
rutas_cliente["cod_cliente"] = rutas_cliente["cod_cliente"].astype(str)

df_merged = pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left"
)


In [ ]:
pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
).cod_ruta.sort_values().unique()

In [ ]:
pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
).groupby("cod_ruta")["Cliente"].nunique()

In [ ]:
pd.merge(
    recomendaciones_para_salesforce,
    rutas_cliente,
    left_on="Cliente",
    right_on="cod_cliente",
    how="left",
).groupby("cod_ruta")["Cliente"].nunique().mean()

In [ ]:
recomendaciones_para_salesforce

In [ ]:
recomendaciones_para_salesforce.Cliente.nunique()

In [ ]:
recomendaciones_para_salesforce.Producto.nunique()

In [ ]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()

**Conteo de marcas recomendadas**

In [ ]:
count = (
    final_rec.groupby("id_cliente")["marca_rec"]
    .apply(lambda x: "-".join((x)))
    .reset_index(name="marcas_ordenadas")
)
count = (
    count.groupby("marcas_ordenadas")["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
)
count

In [ ]:
final_rec.marca_rec.unique()

**Conteo de SKUs recomendadas**

In [ ]:
count2 = (
    final_rec.groupby("id_cliente")["sku"]
    .apply(lambda x: "-".join(x.astype(str)))
    .reset_index(name="marcas_ordenadas")
)
count2 = (
    count2.groupby("marcas_ordenadas")["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
)
count2["f"] = count2["id_cliente"] / count2["id_cliente"].sum()
count2["fa"] = count2["f"].cumsum()
count2

In [ ]:
print(f"RESUMEN para {fecha_tomorrow}")
print(
    "Total de clientes a recomendar: ",
    recomendaciones_para_salesforce.Cliente.nunique(),
)
print("Combinaciones de recomendaciones de MARCAS:", count.shape[0])
print("Marcar recomendadas:", final_rec.marca_rec.unique())
print("Combinaciones de recomendaciones de SKUs:", count2.shape[0])
print(
    "Combinaciones de recomendaciones de SKUs al 80% de clientes:",
    count2[count2["fa"] <= 0.8].shape[0],
)
# print("SKUs disponibles en el historico de todas las rutas:", pan_ventas["cod_articulo_magic"].nunique())
# print("SKUs disponibles en el historico de todas las rutas:", df_ventas["cod_articulo_magic"].nunique())
print("SKUs usados en la recomendacion:", final_rec["sku"].nunique())

In [ ]:
recomendaciones_para_salesforce

In [ ]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()

In [ ]:
asd = (
    recomendaciones_para_salesforce.groupby("Cliente")
    .agg({"Producto": ["count", "nunique"]})
    .reset_index()
)
asd.columns = ["Cliente", "count", "nunique"]
asd["dif"] = asd["count"] == asd["nunique"]
asd.sort_values(["dif", "count"])

In [ ]:
asd.dif.value_counts()

In [ ]:
recomendaciones_para_salesforce

In [ ]:
recomendaciones_para_salesforce.to_csv(
    f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv", index=False
)

In [ ]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Peru/Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(recomendaciones_para_salesforce, s3_path, index=False)

In [ ]:
recomendaciones_para_salesforce.shape

In [ ]:
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)

In [ ]:
# val = pd.merge(recomendaciones_para_salesforce,pan_prod,left_on="Producto",right_on="cod_articulo_magic",how="left")
# val = pd.merge(val,pan_ventas[["cod_compania","cod_cliente","cod_ruta","desc_sucursal","fecha_proceso"]].drop_duplicates(),left_on=["Compania","Cliente"],right_on=["cod_compania","cod_cliente"],how="left")

In [ ]:
# val

In [ ]:
fecha_hoy_val = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=0)
).strftime("%Y-%m-%d")
fecha_hoy_val

In [ ]:
# val.to_csv(f"Val_comercial/D_PE_validacion_recomendaciones_{fecha_hoy_val}.csv",index=False)

In [ ]:
# # Detener el monitoreo
# running = False
# monitor_thread.join()

# # Graficar consumo de RAM y CPU
# fig, ax1 = plt.subplots(figsize=(10, 5))

# ax1.set_xlabel("Tiempo (segundos)")
# ax1.set_ylabel("RAM usada (GB)", color="tab:blue")
# ax1.plot(timestamps, mem_usage_gb, marker="o", linestyle="-", color="tab:blue", label="RAM (GB)")
# ax1.tick_params(axis="y", labelcolor="tab:blue")

# ax2 = ax1.twinx()  # Crear segundo eje Y para CPU
# ax2.set_ylabel("CPU usada (%)", color="tab:red")
# ax2.plot(timestamps, cpu_usage_percent, marker="x", linestyle="--", color="tab:red", label="CPU (%)")
# ax2.tick_params(axis="y", labelcolor="tab:red")

# fig.suptitle("Consumo de CPU y RAM del Notebook")
# ax1.grid()
# fig.tight_layout()
# plt.show()

In [ ]:
# Apaga el Kernel para luego poder usar los demás notebooks sin llenar la memoria
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(False)  # True indica apagar completamente el kernel

# Añadiendo SKUS y CLIENTES MANUALES

In [ ]:
# fecha_tomorrow

In [ ]:
# df = pd.read_csv(f"Output/Peru/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv")

In [ ]:
# # DataFrame inicial con los clientes
# temp_df = pd.DataFrame({"Cliente": [689, 636, 672, 750, 637, 686, 642, 641]})

# # Lista de productos que cada cliente debe tener
# productos = [622396, 622390, 621942, 699204]

# # Expandir el DataFrame para que cada cliente tenga cada uno de los productos
# temp_df = temp_df.loc[temp_df.index.repeat(len(productos))].reset_index(drop=True)

# # Asignar los productos al DataFrame
# temp_df["Producto"] = productos * (len(temp_df) // len(productos))

# temp_df["Pais"] = "PE"
# temp_df["Compania"] = "8013"
# temp_df["Sucursal"] = "01"
# temp_df["Modulo"] = 18766
# temp_df["Cajas"] = int(1)
# temp_df["Unidades"] = int(0)
# temp_df["Fecha"] = f"{fecha_tomorrow}"
# temp_df = temp_df[
#     [
#         "Pais",
#         "Compania",
#         "Sucursal",
#         "Cliente",
#         "Modulo",
#         "Producto",
#         "Cajas",
#         "Unidades",
#         "Fecha",
#     ]
# ]

# temp_df

In [ ]:
# df.shape

In [ ]:
# # Concatenar los DataFrames
# df = pd.concat([df, temp_df], ignore_index=True)

In [ ]:
# df.shape

In [ ]:
# df["Compania"] = df["Compania"].apply(lambda x: str(int(x)).rjust(4, "0"))
# df["Sucursal"] = df["Sucursal"].apply(lambda x: str(int(x)).rjust(2, "0"))

In [ ]:
# df

In [ ]:
# df.groupby(["Compania", "Sucursal"])["Cliente"].nunique()

In [ ]:
# df.to_csv(f"Output/Peru/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv", index=False)

In [ ]:
#